In [1]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

d:\Coding\Projects\hand gesture recognition\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
df = pd.read_csv("final_dataset.csv")
df = df.sample(frac=1).reset_index(drop=True)  # shuffle the dataset
# df = pd.read_csv("cleaned_gesture_data/4Ll3OtqAzyw.csv")

In [3]:
unique_words = set(df['word'])
numbered_unique_words = {word:i for i, word in enumerate(unique_words)}
word_to_number = {i:word for word, i in numbered_unique_words.items()}
df['word'] = df['word'].map(numbered_unique_words)
print(numbered_unique_words)

{'more': 0, 'like': 1, 'hello': 2}


In [19]:
from flask import json


with open("word_mapping.json", "w") as f:
    json.dump(word_to_number, f)

with open("website/word_mapping.json", "w") as f:
    json.dump(word_to_number, f)

In [20]:
testing = df.iloc[0:2, :]
df = df.iloc[2:, :]
print(testing)

   word  row0_frame_index  row0_hand_index  row0_WRIST_x  row0_WRIST_y  \
0     0               0.0              1.0      0.607117      0.555039   
1     0               0.0              0.0      0.306948      0.909797   

   row0_WRIST_z  row0_THUMB_CMC_x  row0_THUMB_CMC_y  row0_THUMB_CMC_z  \
0 -2.664644e-07          0.587633          0.501775          0.001703   
1  1.460996e-07          0.306044          0.846207         -0.001734   

   row0_THUMB_MCP_x  ...  row109_PINKY_MCP_z  row109_PINKY_PIP_x  \
0          0.587644  ...            0.000000            0.000000   
1          0.295868  ...           -0.039522            0.623522   

   row109_PINKY_PIP_y  row109_PINKY_PIP_z  row109_PINKY_DIP_x  \
0            0.000000            0.000000            0.000000   
1            0.721961           -0.045985            0.602045   

   row109_PINKY_DIP_y  row109_PINKY_DIP_z  row109_PINKY_TIP_x  \
0            0.000000            0.000000            0.000000   
1            0.705149     

In [11]:
print(testing.shape, df.shape)

(2, 7151) (10, 7151)


In [5]:
df.iloc[:, 1:].shape

(10, 7150)

In [24]:
model = tf.keras.models.Sequential([
    # tf.keras.layers.Input(shape=(df.shape[0], 3150)),
    tf.keras.layers.Flatten(input_shape=(df.iloc[:, 1:].shape[1],)),
    tf.keras.layers.Dense(1024, activation="relu"),
    tf.keras.layers.Dense(1024, activation="relu"),
    tf.keras.layers.Dense(512, activation="relu"),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(numbered_unique_words), activation='softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
history = model.fit(df.iloc[:, 1:], df['word'], epochs=50)

Epoch 1/50


d:\Coding\Projects\hand gesture recognition\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 801ms/step - accuracy: 0.2000 - loss: 1.9702
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5000 - loss: 3.8957
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.3000 - loss: 6.9228
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7000 - loss: 2.7677
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.5000 - loss: 8.0091
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5000 - loss: 1.3486
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2000 - loss: 7.3530
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.3000 - loss: 3.7079
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.3000 - loss: 1.9657
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.7000 - loss: 0.8835
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.5000 - loss: 1.4745
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.7000 - loss: 1.0245
Epoch 13/50

In [25]:
pred = model.predict(testing.iloc[0, 1:].values.reshape(1, df.iloc[:, 1:].shape[1]))
print(testing.iloc[0][0])
print(word_to_number[np.argmax(pred, axis=1)[0]])
pred = model.predict(testing.iloc[1, 1:].values.reshape(1, df.iloc[:, 1:].shape[1]))
print(testing.iloc[1][0])
print(word_to_number[np.argmax(pred, axis=1)[0]])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
0.0
more
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
0.0
more


C:\Users\Nathan Apte\AppData\Local\Temp\ipykernel_24156\1341466554.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(testing.iloc[0][0])
C:\Users\Nathan Apte\AppData\Local\Temp\ipykernel_24156\1341466554.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(testing.iloc[1][0])


In [27]:
model.save("hand_gesture_model2.keras")
model.save("website/hand_gesture_model2.keras")

In [26]:
test_set = pd.read_csv("landmark_data.csv", header=None)
# model = tf.keras.models.load_model("hand_gesture_model.keras")
pred = model.predict(test_set)
print(np.max(pred, axis=1))
print(np.argmax(pred, axis=1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[0.94806725]
[2]
